In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# ============================================================
# BANKING DATA ENGINEERING PROJECT
# BRONZE LAYER
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

import pandas as pd


# ============================================================
# 1. SPARK SESSION
# ============================================================

spark = SparkSession.builder.appName(
    "Banking_Bronze"
).getOrCreate()


# ============================================================
# 2. SOURCE CONFIGURATION
# ============================================================

source_path = "/Volumes/bank_data/default/bank_data/bank.xlsx"

source_file = "bank.xlsx"

bronze_table = "bronze_bank_transactions"


# ============================================================
# 3. READ EXCEL FILE
# ============================================================

print("========================================")
print("         RAW FILE INSPECTION")
print("========================================")

raw_df = pd.read_excel(
    source_path,
    sheet_name="Sheet1",
    engine="openpyxl"
)

print(f"Rows: {raw_df.shape[0]}")
print(f"Columns: {raw_df.shape[1]}")

print("\nColumn Names:")
print(raw_df.columns.tolist())

print("\nData Types:")
print(raw_df.dtypes)

print("\nMissing Values:")
print(raw_df.isnull().sum())


# ============================================================
# 4. RENAME COLUMNS
# ============================================================

raw_df = raw_df.rename(columns={
    "Account No": "Account_No",
    "TRANSACTION DETAILS": "TRANSACTION_DETAILS",
    "VALUE DATE": "VALUE_DATE",
    "WITHDRAWAL AMT": "WITHDRAWAL_AMT",
    "DEPOSIT AMT": "DEPOSIT_AMT",
    "BALANCE AMT": "BALANCE_AMT"
})


# ============================================================
# 5. CONVERT DATES TO STRINGS
# ============================================================
# Bronze keeps raw date values as strings.
# Date conversion will happen in the Silver layer.

raw_df["DATE"] = raw_df["DATE"].apply(
    lambda x: x.strftime("%Y-%m-%d %H:%M:%S")
    if pd.notna(x)
    else None
)

raw_df["VALUE_DATE"] = raw_df["VALUE_DATE"].apply(
    lambda x: x.strftime("%Y-%m-%d %H:%M:%S")
    if pd.notna(x)
    else None
)


# ============================================================
# 6. STANDARDIZE STRING COLUMNS
# ============================================================

raw_df["Account_No"] = raw_df["Account_No"].apply(
    lambda x: str(x) if pd.notna(x) else None
)

raw_df["TRANSACTION_DETAILS"] = raw_df[
    "TRANSACTION_DETAILS"
].apply(
    lambda x: str(x) if pd.notna(x) else None
)

raw_df["."] = raw_df["."].apply(
    lambda x: str(x) if pd.notna(x) else None
)


# ============================================================
# 7. STANDARDIZE NUMERIC COLUMNS
# ============================================================

raw_df["CHQ.NO."] = pd.to_numeric(
    raw_df["CHQ.NO."],
    errors="coerce"
)

raw_df["WITHDRAWAL_AMT"] = pd.to_numeric(
    raw_df["WITHDRAWAL_AMT"],
    errors="coerce"
)

raw_df["DEPOSIT_AMT"] = pd.to_numeric(
    raw_df["DEPOSIT_AMT"],
    errors="coerce"
)

raw_df["BALANCE_AMT"] = pd.to_numeric(
    raw_df["BALANCE_AMT"],
    errors="coerce"
)


# ============================================================
# 8. REPLACE NaN WITH NONE
# ============================================================

raw_df = raw_df.astype(object)

raw_df = raw_df.where(
    pd.notna(raw_df),
    None
)


# ============================================================
# 9. EXPLICIT SPARK SCHEMA
# ============================================================
# DATE and VALUE_DATE are intentionally STRING in Bronze.
# They will be converted properly in Silver.

schema = StructType([

    StructField(
        "Account_No",
        StringType(),
        True
    ),

    StructField(
        "DATE",
        StringType(),
        True
    ),

    StructField(
        "TRANSACTION_DETAILS",
        StringType(),
        True
    ),

    StructField(
        "CHQ.NO.",
        DoubleType(),
        True
    ),

    StructField(
        "VALUE_DATE",
        StringType(),
        True
    ),

    StructField(
        "WITHDRAWAL_AMT",
        DoubleType(),
        True
    ),

    StructField(
        "DEPOSIT_AMT",
        DoubleType(),
        True
    ),

    StructField(
        "BALANCE_AMT",
        DoubleType(),
        True
    ),

    StructField(
        ".",
        StringType(),
        True
    )
])


# ============================================================
# 10. CONVERT PANDAS → PYTHON RECORDS
# ============================================================

records = [
    tuple(row)
    for row in raw_df.itertuples(
        index=False,
        name=None
    )
]

print("\n========================================")
print("         DATA CONVERSION")
print("========================================")

print(
    f"Records prepared for Spark: {len(records)}"
)


# ============================================================
# 11. CREATE SPARK DATAFRAME
# ============================================================

bronze_df = spark.createDataFrame(
    records,
    schema=schema
)


# ============================================================
# 12. ADD BRONZE METADATA
# ============================================================

bronze_df = (
    bronze_df
    .withColumn(
        "_source_file",
        lit(source_file)
    )
    .withColumn(
        "_ingestion_timestamp",
        current_timestamp()
    )
)


# ============================================================
# 13. WRITE BRONZE DELTA TABLE
# ============================================================

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        bronze_table
    )
)


# ============================================================
# 14. BRONZE VALIDATION
# ============================================================

expected_rows = 116201

expected_columns = 11

actual_rows = bronze_df.count()

actual_columns = len(
    bronze_df.columns
)


print("\n========================================")
print("         BRONZE VALIDATION")
print("========================================")

print(
    f"Expected Rows: {expected_rows}"
)

print(
    f"Actual Rows: {actual_rows}"
)


if actual_rows == expected_rows:

    print(
        "BRONZE INGESTION PASSED"
    )

else:

    print(
        "BRONZE INGESTION FAILED"
    )


print(
    f"\nExpected Columns: {expected_columns}"
)

print(
    f"Actual Columns: {actual_columns}"
)


if actual_columns == expected_columns:

    print(
        "COLUMN COUNT VALIDATION PASSED"
    )

else:

    print(
        "COLUMN COUNT VALIDATION FAILED"
    )


# ============================================================
# 15. BRONZE SCHEMA
# ============================================================

print("\n========================================")
print("         BRONZE SCHEMA")
print("========================================")

bronze_df.printSchema()


# ============================================================
# 16. BRONZE SAMPLE DATA
# ============================================================

print("\n========================================")
print("         BRONZE SAMPLE DATA")
print("========================================")

display(
    bronze_df.limit(10)
)


# ============================================================
# 17. TABLE VALIDATION
# ============================================================

print("\n========================================")
print("         TABLE VALIDATION")
print("========================================")

table_exists = spark.catalog.tableExists(
    bronze_table
)


if table_exists:

    print(
        f" Table exists: {bronze_table}"
    )

else:

    print(
        f"Table does not exist: {bronze_table}"
    )


# ============================================================
# 18. FINAL BRONZE CHECK
# ============================================================

if (
    actual_rows == expected_rows
    and actual_columns == expected_columns
    and table_exists
):

    print("\n========================================")
    print("     BRONZE LAYER COMPLETED")
    print("========================================")

    print(
        " ALL BRONZE CHECKS PASSED"
    )

    print(
        f"Bronze Table: {bronze_table}"
    )

    print(
        f"Rows: {actual_rows}"
    )

    print(
        f"Columns: {actual_columns}"
    )

    print(
        f"Source File: {source_file}"
    )

else:

    raise Exception(
        " BRONZE DATA QUALITY CHECK FAILED"
    )

         RAW FILE INSPECTION
Rows: 116201
Columns: 9

Column Names:
['Account No', 'DATE', 'TRANSACTION DETAILS', 'CHQ.NO.', 'VALUE DATE', 'WITHDRAWAL AMT', 'DEPOSIT AMT', 'BALANCE AMT', '.']

Data Types:
Account No                     object
DATE                   datetime64[ns]
TRANSACTION DETAILS            object
CHQ.NO.                       float64
VALUE DATE             datetime64[ns]
WITHDRAWAL AMT                float64
DEPOSIT AMT                   float64
BALANCE AMT                   float64
.                              object
dtype: object

Missing Values:
Account No                  0
DATE                        0
TRANSACTION DETAILS      2499
CHQ.NO.                115296
VALUE DATE                  0
WITHDRAWAL AMT          62652
DEPOSIT AMT             53549
BALANCE AMT                 0
.                           0
dtype: int64

         DATA CONVERSION
Records prepared for Spark: 116201

         BRONZE VALIDATION
Expected Rows: 116201
Actual Rows: 116201
BRONZE I

Account_No,DATE,TRANSACTION_DETAILS,CHQ.NO.,VALUE_DATE,WITHDRAWAL_AMT,DEPOSIT_AMT,BALANCE_AMT,.,_source_file,_ingestion_timestamp
409000611074',2017-06-29 00:00:00,TRF FROM Indiaforensic SERVICES,null,2017-06-29 00:00:00,null,1000000.0,1000000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-07-05 00:00:00,TRF FROM Indiaforensic SERVICES,null,2017-07-05 00:00:00,null,1000000.0,2000000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-07-18 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-07-18 00:00:00,null,500000.0,2500000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-01 00:00:00,TRF FRM Indiaforensic SERVICES,null,2017-08-01 00:00:00,null,3000000.0,5500000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,6000000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,6500000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,7000000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,7500000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,8000000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z
409000611074',2017-08-16 00:00:00,FDRL/INTERNAL FUND TRANSFE,null,2017-08-16 00:00:00,null,500000.0,8500000.0,.,bank.xlsx,2026-10-06T21:02:31.671Z



         TABLE VALIDATION
 Table exists: bronze_bank_transactions

     BRONZE LAYER COMPLETED
 ALL BRONZE CHECKS PASSED
Bronze Table: bronze_bank_transactions
Rows: 116201
Columns: 11
Source File: bank.xlsx
